In [1]:
!pip install -q youtube-transcript-api langchain faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 32.9 MB/s eta 0:00:00


In [2]:
import os
import google.generativeai as genai

# Set your API Key here
os.environ["GOOGLE_API_KEY"] = "API"

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
from youtube_transcript_api import YouTubeTranscriptApi

def get_transcript(video_id):
    transcript = YouTubeTranscriptApi.get_transcript(video_id)
    full_text = " ".join([t["text"] for t in transcript])
    return full_text

In [12]:
video_id = "CnXdddeZ4tQ" #here we can replace this ID with any other YT Video Id

try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

Companies like Uber, Rapolit, LinkedIn uses a framework called Langraph to build agents that are not just powerful but reliable. There are so many agent AI frameworks out there and Langraph is one of the popular framework. In this particular crash course, we will look at some theoretical fundamentals first and then we will dive deeper into langraph covering a range of topics which I have displayed on the screen right now. I also want to take a moment to thank Jad Brains for sponsoring this video. As part of this collaboration, you all will get 3 month free subscription of PyCharm Pro. More details on that later. Let's start with agentic AI fundamentals through Langraph. We will be building AI agents. So, it's important you know what it is. And if you already know this concept, feel free to skip to the next section. When you have a simple LLM and when you ask a question, it can give limited answer. based on its training cutoff. So let's say if you ask questions specific to your organiza

In [13]:
!pip install langchain_community

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    return splitter.create_documents([text])

In [15]:
from langchain.embeddings.base import Embeddings
import google.generativeai as genai

class GeminiEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [
            genai.embed_content(
                model="gemini-embedding-001",
                content=text
            )["embedding"]
            for text in texts
        ]

    def embed_query(self, text):
        return genai.embed_content(
            model="gemini-embedding-001",
            content=text
        )["embedding"]

In [16]:
from langchain_community.vectorstores import FAISS

# Initialize embeddings
embeddings = GeminiEmbeddings()

# Split the transcript into chunks
chunks = split_text(transcript)

# Create vector DB
vectorstore = FAISS.from_documents(chunks, embeddings)

In [17]:
def get_retriever(vectorstore):
    return vectorstore.as_retriever(search_kwargs={"k": 3})

In [18]:
def rag_query(query, retriever):
    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
    Answer ONLY from the context below.

    Context:
    {context}

    Question:
    {query}
    """

    model = genai.GenerativeModel("gemini-2.5-flash")
    response = model.generate_content(prompt)

    return response.text

In [19]:
print(rag_query("Summarize the video", get_retriever(vectorstore)))
print(rag_query("What are the key concepts discussed?", get_retriever(vectorstore)))

The context discusses a system that manages state, including resuming complex states for error recovery or human workflow, using checkpoints. It illustrates how humans can be involved in agent AI workflows through "interrupt and command," such as approving a stock purchase ("you bought 10 stocks") or declining it ("buying decline"). The system uses an LLM to enhance output and smartly determine when to call a tool (e.g., to retrieve stock prices for Apple, Amazon, or MSFT, and calculate total costs) versus when not to (e.g., answering "who invented the theory of relativity"). It highlights how calculations for stock purchases (e.g., 5 Apple stocks + 5 MSFT stocks totaling $1503) sometimes do not work as expected. Memory is used to maintain context across different threads.
The key concepts discussed are:

*   **Maintaining Thread Context:** The system maintains separate contexts for different threads (e.g., thread one, thread two, config 1, config 2), allowing it to remember specific c

In [20]:
print(rag_query("What is this video all about, what is the topic being discussed?", get_retriever(vectorstore)))

The video is about agent AI workflows and how humans can be involved in them using the interrupt and command. It discusses Anthropic's definition of AI systems built using LLMs, categorizing them into workflows and agents. The topic is specifically about learning agentic AI using Langraph, using examples like retrieving stock prices and handling complex stock purchase requests.
